In [1]:
!rm -r xCaliber

In [2]:
!git clone https://github.com/Pranshu-Bahadur/xCaliber.git

Cloning into 'xCaliber'...
remote: Enumerating objects: 754, done.
remote: Counting objects: 100% (283/283), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 754 (delta 124), reused 243 (delta 87), pack-reused 471 (from 1)
Receiving objects: 100% (754/754), 20.93 MiB | 32.83 MiB/s, done.
Resolving deltas: 100% (302/302), done.


In [3]:
%pip install ninja

In [4]:
import os

os.environ['TORCH_CUDA_ARCH_LIST'] = "7.5"

In [14]:
%cd xCaliber

/content/xCaliber


In [5]:
!TORCH_SHOW_CPP_DETAILS=1 python ./xCaliber/xcaliber/setup.py build_ext --inplace

running build_ext
building 'xcalibur' extension
[1/2] /usr/local/cuda/bin/nvcc -MD -MF /content/build/temp.linux-x86_64-cpython-313/content/xCaliber/xcaliber/inference/fused-moe/moe.o.d -I/usr/local/lib/python3.13/dist-packages/torch/include -I/usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -I/usr/local/cuda/include -I/usr/include/python3.13 -c -c /content/xCaliber/xcaliber/inference/fused-moe/moe.cu -o /content/build/temp.linux-x86_64-cpython-313/content/xCaliber/xcaliber/inference/fused-moe/moe.o -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ --expt-relaxed-constexpr --compiler-options ''"'"'-fPIC'"'"'' -O3 --use_fast_math -std=c++17 -DTORCH_API_INCLUDE_EXTENSION_H -DTORCH_EXTENSION_NAME=xcalibur -gencode=arch=compute_75,code=sm_75
[2/2] c++ -MMD -MF /content/build/temp.linux-x86_64-cpython-313/content/xCaliber/xcaliber/inference/fused-moe/bindings.o.d -fno-strict-overflow -Wsig

In [24]:
import sys
!{sys.executable} -m pytest ./xCaliber/test/test_moe.py

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: typeguard-4.6.0, langsmith-0.12.1, anyio-4.14.2
collected 1 item                                                               

xCaliber/test/test_moe.py F                                              [100%]

=================================== FAILURES ===================================
__________________________________ test_topk ___________________________________

    def test_topk():
        print(f"GPU: {torch.cuda.get_device_name()}")
        print("     N     E   K activation  xcalibur_us    torch_us  speedup")
        # The current binding launches on the default stream.
        with torch.cuda.stream(torch.cuda.default_stream()):
            for softmax in (False, True):
                for N in (8, 16, 16384):
                    for E in (256, 512, 1024):
                        for K in (2, 8):
>         